<a href="https://colab.research.google.com/github/Djerad/Tranformers/blob/main/Text%20Summarizer_BART_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install transformers gradio torch --quiet

In [9]:
!pip install -U transformers accelerate sentencepiece --quiet

In [10]:
!pip install "transformers==4.44.2" --quiet


In [16]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=device,
)
print("✅ Model ready!")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready!


In [24]:
def summarize(text: str, length_preset: str = "Medium", num_beams: int = 4):
    text = text.strip()

    if not text:
        return "", "⚠️ Empty text"

    word_count = len(text.split())

    if word_count < 30:
        return "", "⚠️ Text too short"

    presets = {
        "Short": (30, 80),
        "Medium": (80, 180),
        "Long": (180, 350),
    }

    min_len, max_len = presets.get(length_preset, (50, 150))

    max_len = min(max_len, word_count - 1)
    min_len = max(10, min_len)

    result = summarizer(
        text,
        min_length=min_len,
        max_length=max_len,
        do_sample=False,               # 🔥 important fix
        truncation=True,               # 🔥 prevents overflow
        num_beams=num_beams,
        no_repeat_ngram_size=3,
        length_penalty=2.0,
        early_stopping=True,
    )

    summary = result[0]["summary_text"]

    summary_words = len(summary.split())
    compression = round((1 - summary_words / word_count) * 100, 1)

    info = f"Input: {word_count} words → Summary: {summary_words} words ({compression}% compression)"

    return summary, info

In [25]:
def chunk_text(text, max_words=700):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]


def summarize_long(text):
    chunks = chunk_text(text)
    results = []

    for chunk in chunks:
        res = summarizer(
            chunk,
            max_length=150,
            min_length=50,
            do_sample=False,
            truncation=True,
        )
        results.append(res[0]["summary_text"])

    return " ".join(results)

In [26]:
import gradio as gr

with gr.Blocks(title="Text Summarizer 🧠") as demo:

    gr.Markdown("# 🧠 AI Text Summarizer")

    with gr.Row():
        input_text = gr.Textbox(
            lines=18,
            placeholder="Paste your text here...",
            label="Input Text"
        )

        output_text = gr.Textbox(
            lines=18,
            label="Summary",
            interactive=False
        )

    info = gr.Markdown()

    with gr.Row():
        length = gr.Radio(
            ["Short", "Medium", "Long"],
            value="Medium",
            label="Summary Length"
        )

        beams = gr.Slider(
            1, 8, value=4, step=1,
            label="Beam Search"
        )

    with gr.Row():
        btn = gr.Button("🚀 Summarize")
        clear = gr.Button("🧹 Clear")

    # -----------------------
    # Function wrapper (safe)
    # -----------------------
    def run_summarize(text, length, beams):
        try:
            summary, info_text = summarize(text, length, beams)
            return summary, info_text
        except Exception as e:
            return "", f"❌ Error: {str(e)}"

    btn.click(
        run_summarize,
        inputs=[input_text, length, beams],
        outputs=[output_text, info]
    )

    def clear_all():
        return "", "", ""

    clear.click(
        clear_all,
        outputs=[input_text, output_text, info]
    )

    # -----------------------
    # Examples (VERY useful)
    # -----------------------
    gr.Examples(
        examples=[
            ["Artificial intelligence is transforming the world rapidly...", "Medium", 4],
            ["Space exploration has advanced significantly in the last decade...", "Short", 3],
        ],
        inputs=[input_text, length, beams]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2227285a64ba64ad39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2227285a64ba64ad39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
